<a href="https://colab.research.google.com/github/Joan2022Laurente/fade/blob/main/notebooks/Qwen-Image-2512T2I.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Git clone the repo and install the requirements. (ignore the pip errors about protobuf)

In [ ]:
# ================================================================
#   CELDA ÚNICA DE SETUP — Qwen-Image-2512 T2I WORKFLOW
#   Para Google Colab · GPU ≥15 GB VRAM · GGUF Q4_K_M
# ================================================================

import os, subprocess, sys, shutil
from google.colab import userdata

# ── SECRETS ────────────────────────────────────────────────────
HF_TOKEN      = userdata.get('HF_TOKEN')
TAILSCALE_KEY = userdata.get('TAILSCALE_KEY')

if not HF_TOKEN:
    raise RuntimeError("❌ Falta HF_TOKEN en Secrets de Colab")
if not TAILSCALE_KEY:
    raise RuntimeError("❌ Falta TAILSCALE_KEY en Secrets de Colab")

os.environ['HF_TOKEN']      = HF_TOKEN
os.environ['TAILSCALE_KEY'] = TAILSCALE_KEY
print("✅ Secrets cargados")

# ──────────────────────────────────────────────────────────────
# CONFIGURACIÓN
# ──────────────────────────────────────────────────────────────
COMFY         = "/content/ComfyUI"
CUSTOM_NODES  = f"{COMFY}/custom_nodes"
MODELS        = f"{COMFY}/models"
WORKFLOWS_DST = f"{COMFY}/user/default/workflows"
WORKFLOW_SRC  = "/content/workflowT2I.json"

HF_COMFY     = "https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files"
HF_UNSLOTH   = "https://huggingface.co/unsloth/Qwen-Image-2512-GGUF/resolve/main"
HF_LIGHTNING = "https://huggingface.co/lightx2v/Qwen-Image-2512-Lightning/resolve/main"

MODELS_TO_DOWNLOAD = [
    (f"{HF_UNSLOTH}/qwen-image-2512-Q4_K_M.gguf",                              "unet",          "qwen-image-2512-Q4_K_M.gguf",                             False),
    (f"{HF_COMFY}/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors",        "text_encoders", "qwen_2.5_vl_7b_fp8_scaled.safetensors",                   False),
    (f"{HF_COMFY}/vae/qwen_image_vae.safetensors",                             "vae",           "qwen_image_vae.safetensors",                               False),
    (f"{HF_LIGHTNING}/Qwen-Image-2512-Lightning-4steps-V1.0-bf16.safetensors", "loras",         "Qwen-Image-2512-Lightning-4steps-V1.0-bf16.safetensors",   False),
    ("https://huggingface.co/joanjlau/loraprueba0/resolve/main/YummyHDqwen.safetensors",
                                                                                "loras",         "YummyHDqwen.safetensors",                                  True),
]

# ──────────────────────────────────────────────────────────────
# HELPERS
# ──────────────────────────────────────────────────────────────
def run(cmd, cwd=None, check=True):
    print(f"  $ {cmd[:120]}")
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = r.stdout.strip()
    if out:
        print(out[-2000:])
    if check and r.returncode != 0:
        print(f"  ⚠️  Exit code {r.returncode} — continuando de todos modos")
    return r.returncode

def pip(pkg):
    run(f'"{sys.executable}" -m pip install {pkg} -q --no-deps 2>/dev/null || '
        f'"{sys.executable}" -m pip install {pkg} -q', check=False)

def clone_or_update(url, dst, name):
    if os.path.isdir(dst):
        print(f"  ↻  {name} ya existe — actualizando")
        run("git pull --ff-only", cwd=dst, check=False)
    else:
        run(f'git clone --depth 1 "{url}" "{dst}"')
    req = os.path.join(dst, "requirements.txt")
    if os.path.exists(req):
        run(f'"{sys.executable}" -m pip install -r "{req}" -q', check=False)

def download_hf(url, dest_subdir, filename, private=False):
    dest_dir = os.path.join(MODELS, dest_subdir)
    os.makedirs(dest_dir, exist_ok=True)
    dest = os.path.join(dest_dir, filename)
    if os.path.exists(dest) and os.path.getsize(dest) > 1024 * 1024:
        gb = os.path.getsize(dest) / 1024**3
        print(f"  ✅ {filename} ya existe ({gb:.2f} GB) — omitiendo")
        return
    print(f"  ↓  Descargando {filename} ...")
    header = f'--header="Authorization: Bearer {HF_TOKEN}"' if private else ''
    ret = run(
        f'aria2c -c -x 8 -s 8 -k 5M --file-allocation=none '
        f'--console-log-level=warn {header} '
        f'"{url}" -d "{dest_dir}" -o "{filename}"',
        check=False
    )
    if ret != 0 or not os.path.exists(dest) or os.path.getsize(dest) < 1024 * 1024:
        print("  → aria2c falló, intentando con wget...")
        auth = f'--header="Authorization: Bearer {HF_TOKEN}"' if private else ''
        run(f'wget -q --show-progress {auth} -O "{dest}" "{url}"', check=False)
    if os.path.exists(dest) and os.path.getsize(dest) > 1024 * 1024:
        gb = os.path.getsize(dest) / 1024**3
        print(f"  ✅ {filename} ({gb:.2f} GB)")
    else:
        print(f"  ❌ FALLO al descargar {filename}")

# ──────────────────────────────────────────────────────────────
print("=" * 66)
print("  SETUP — Qwen-Image-2512 T2I · GGUF Q4_K_M + Lightning 4s BF16")
print("=" * 66)

print("\n[SYS] Instalando herramientas del sistema...")
run("apt-get update -qq && apt-get install -y -qq aria2 git wget curl", check=False)

print("\n[0/5] ComfyUI base...")
if not os.path.isdir(COMFY):
    run(f'git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git "{COMFY}"')
else:
    print("  ✅ ComfyUI ya existe")
    run("git pull --ff-only", cwd=COMFY, check=False)

if os.path.exists(f"{COMFY}/requirements.txt"):
    run(f'"{sys.executable}" -m pip install -r "{COMFY}/requirements.txt" -q', check=False)

for d in [
    CUSTOM_NODES, WORKFLOWS_DST,
    f"{MODELS}/unet", f"{MODELS}/diffusion_models",
    f"{MODELS}/text_encoders", f"{MODELS}/vae",
    f"{MODELS}/loras", f"{MODELS}/checkpoints", f"{MODELS}/clip",
]:
    os.makedirs(d, exist_ok=True)

print("\n[1/5] ComfyUI-GGUF (UnetLoaderGGUF)...")
clone_or_update("https://github.com/city96/ComfyUI-GGUF.git", f"{CUSTOM_NODES}/ComfyUI-GGUF", "ComfyUI-GGUF")
pip("gguf")

print("\n[2/5] rgthree-comfy...")
clone_or_update("https://github.com/rgthree/rgthree-comfy.git", f"{CUSTOM_NODES}/rgthree-comfy", "rgthree-comfy")

print("\n[3/5] cg-use-everywhere...")
clone_or_update("https://github.com/chrisgoringe/cg-use-everywhere.git", f"{CUSTOM_NODES}/cg-use-everywhere", "cg-use-everywhere")

print("\n[4/5] Dependencias Python...")
for pkg in [
    "transformers>=4.51.3", "accelerate", "einops", "torchvision",
    "opencv-python-headless", "Pillow", "scipy", "scikit-image", "gguf", "timm",
]:
    pip(pkg)
print("  ✅ Paquetes instalados")

print("\n[5/5] Descargando modelos...")
for url, subfolder, filename, private in MODELS_TO_DOWNLOAD:
    download_hf(url, subfolder, filename, private)

print("\nCopiando workflow JSON a ComfyUI...")
if os.path.exists(WORKFLOW_SRC):
    dst = f"{WORKFLOWS_DST}/workflowT2I.json"
    shutil.copy2(WORKFLOW_SRC, dst)
    print(f"  ✅ Workflow copiado → {dst}")
else:
    print(f"  ⚠️  No se encontró {WORKFLOW_SRC}")
    print(f"     → Sube workflowT2I.json a /content/ y re-ejecuta")

print("\n" + "─" * 66)
print("  📂  SUBE TU LoRA aquí antes de generar:")
print(f"      {MODELS}/loras/")
print("  En el workflow: nodo '🎨 TU LORA'  →  selecciona el archivo")
print("  Peso sugerido: 0.85  (baja a 0.5 si hay artefactos/gridlines)")
print("─" * 66)

print("\n" + "=" * 66)
print("   VERIFICACIÓN DE ARCHIVOS")
print("=" * 66)

LIGHTNING_LORA = "Qwen-Image-2512-Lightning-4steps-V1.0-bf16.safetensors"
checks = [
    (f"{MODELS}/unet/qwen-image-2512-Q4_K_M.gguf",                    "Modelo GGUF Q4_K_M"),
    (f"{MODELS}/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors", "Text Encoder FP8"),
    (f"{MODELS}/vae/qwen_image_vae.safetensors",                       "VAE"),
    (f"{MODELS}/loras/{LIGHTNING_LORA}",                               "Lightning 4-step BF16"),
    (f"{MODELS}/loras/YummyHDqwen.safetensors",                        "LoRA privado YummyHD"),
    (f"{CUSTOM_NODES}/ComfyUI-GGUF",                                   "GGUF Loader node"),
    (f"{CUSTOM_NODES}/rgthree-comfy",                                  "rgthree"),
    (f"{CUSTOM_NODES}/cg-use-everywhere",                              "Use Everywhere"),
]

for path, label in checks:
    exists    = os.path.exists(path)
    too_small = exists and os.path.isfile(path) and os.path.getsize(path) < 1024 * 1024
    ok = exists and not too_small
    icon = "✅" if ok else "❌"
    print(f"  {icon}  {label:<32} {path.split('/')[-1]}")

print("\n" + "=" * 66)
print("  ✅  SETUP COMPLETO")
print("  → Lanza ComfyUI y carga workflowT2I.json")
print("  → KSampler: Steps=4 · CFG=1.0 · sampler=euler · scheduler=beta")
print("=" * 66)

✅ Secrets cargados
  SETUP — Qwen-Image-2512 T2I · GGUF Q4_K_M + Lightning 4s BF16

[SYS] Instalando herramientas del sistema...
  $ apt-get update -qq && apt-get install -y -qq aria2 git wget curl
2_1.36.0-1_amd64.deb ...
Unpacking aria2 (1.36.0-1) ...
Preparing to unpack .../3-libcurl4-openssl-dev_7.81.0-1ubuntu1.24_amd64.deb ...
Unpacking libcurl4-openssl-dev:amd64 (7.81.0-1ubuntu1.24) over (7.81.0-1ubuntu1.23) ...
Preparing to unpack .../4-curl_7.81.0-1ubuntu1.24_amd64.deb ...
Unpacking curl (7.81.0-1ubuntu1.24) over (7.81.0-1ubuntu1.23) ...
Preparing to unpack .../5-libcurl4_7.81.0-1ubuntu1.24_amd64.deb ...
Unpacking libcurl4:amd64 (7.81.0-1ubuntu1.24) over (7.81.0-1ubuntu1.23) ...
Setting up libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Setting up libcurl4:amd64 (7.81.0-1ubuntu1.24) ...
Setting up curl (7.81.0-1ubuntu1.24) ...
Setting up libaria2-0:amd64 (1.36.0-1) ...
Setting up aria2 (1.36.0-1) ...
Setting up libcurl4-openssl-dev:amd64 (7.81.0-1ubuntu1.24) ...
Processing trigg

In [ ]:
# 🔒 Lanzar ComfyUI con Tailscale

import os, time
from google.colab import userdata

TAILSCALE_AUTH_KEY = userdata.get('TAILSCALE_KEY')
if not TAILSCALE_AUTH_KEY:
    raise RuntimeError("❌ Falta TAILSCALE_KEY en Secrets de Colab")

# 1. Instalar Tailscale
!curl -fsSL https://tailscale.com/install.sh | sh

# 2. Iniciar daemon
print("🚀 Iniciando tailscaled...")
!nohup tailscaled --tun=userspace-networking --socks5-server=localhost:1055 > /tmp/tailscaled.log 2>&1 &
time.sleep(5)

# 3. Verificar
print("🔍 Verificando tailscaled...")
!ps aux | grep tailscaled

# 4. Conectar
print("🔗 Conectando a Tailscale...")
!tailscale up --authkey={TAILSCALE_AUTH_KEY} --hostname=colab-comfyui

# 5. Mostrar IP
print("\n" + "="*50)
print("🌐 TU IP PRIVADA DE TAILSCALE ES:")
!tailscale ip -4
print("="*50)

# 6. Lanzar ComfyUI
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print("\n🚀 Iniciando ComfyUI...")
%cd /content/ComfyUI
!python main.py --listen 0.0.0.0 --dont-print-server --lowvram

Installing Tailscale for ubuntu jammy, using method apt
+ mkdir -p --mode=0755 /usr/share/keyrings
+ curl -fsSL https://pkgs.tailscale.com/stable/ubuntu/jammy.noarmor.gpg
+ tee /usr/share/keyrings/tailscale-archive-keyring.gpg
+ chmod 0644 /usr/share/keyrings/tailscale-archive-keyring.gpg
+ curl -fsSL https://pkgs.tailscale.com/stable/ubuntu/jammy.tailscale-keyring.list
+ tee /etc/apt/sources.list.d/tailscale.list
# Tailscale packages for ubuntu jammy
deb [signed-by=/usr/share/keyrings/tailscale-archive-keyring.gpg] https://pkgs.tailscale.com/stable/ubuntu jammy main
+ chmod 0644 /etc/apt/sources.list.d/tailscale.list
+ apt-get update
Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:6 https://cloud.r-project.org/bin/linux/

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 🔒 Lanzar ComfyUI con Cloudflare Tunnel

import os
import subprocess
import threading
import time

# --- CONFIGURACIÓN E INSTALACIÓN ---

print("📦 Instalando Cloudflared...")
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# --- FUNCIÓN PARA EL TÚNEL ---

def run_cloudflare():
    # Creamos el túnel apuntando al puerto 8188 (por defecto de ComfyUI)
    p = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    # Buscamos la URL generada en los logs
    for line in p.stdout:
        if "trycloudflare.com" in line:
            print("\n" + "="*60)
            print("🌐 TU URL PÚBLICA ES:")
            print(line.strip().split(" ")[-1])
            print("="*60 + "\n")
        # Si quieres ver los logs de cloudflare, descomenta la siguiente línea:
        # print(line, end="")

# 1. Iniciar el túnel en segundo plano
threading.Thread(target=run_cloudflare, daemon=True).start()

# 2. Esperar un momento a que el túnel se establezca
time.sleep(5)

# 3. Lanzar ComfyUI
print("🚀 Iniciando ComfyUI...")
%cd /content/ComfyUI
# Usamos 127.0.0.1 porque el túnel de Cloudflare está escuchando localmente
!python main.py --dont-print-server